# Parse Transfer Capacities

Preparation of transfer capacities. Currently, we take the old values from winter 2010/2011 and convert them to hourly values to be consistent with later approaches

jab 01.06.2019

script checked by Jonas 23.07.2025: works but need to check in create gdx if still needed

Jonas 16.09.2025, added new NTC data: https://2024.entsos-tyndp-scenarios.eu/download/

## Packages and options

In [12]:
import pandas as pd
import numpy as np

In [13]:
fn_in = "../source_data/20231103 - Electricity and Hydrogen Reference Grid & Investment Candidates.xlsx"
fn_out = "../parsed_data/ntc.csv"

In [14]:
dict_agg_country = {
    'AT00':'AT',
'BE00':'BE',
'BG00':'BG',
'CH00':'CH',
'CZ00':'CZ',
'DE00':'DE',
'DEKF':'DE',
'DKE1':'DK',
'DKKF':'DK',
'DKW1':'DK',
'EE00':'EE',
'ES00':'ES',
'FI00':'FI',
'FR00':'FR',
'FR15':'FR',
'GR00':'GR',
'GR03':'GR',
'HR00':'HR',
'HU00':'HU',
'IE00':'IE',
'ITCA':'IT',
'ITCN':'IT',
'ITCO':'IT',
'ITCS':'IT',
'ITN1':'IT',
'ITS1':'IT',
'ITSA':'IT',
'ITSI':'IT',
'ITSIvirt':'IT',
'LT00':'LT',
'LUB1':'LU',
'LUF1':'LU',
'LUG1':'LU',
'LUV1':'LU',
'LV00':'LV',
'NL00':'NL',
'NOM1':'NO',
'NON1':'NO',
'NOS0':'NO',
'PL00':'PL',
'PL00E':'PL',
'PL00I':'PL',
'PT00':'PT',
'RO00':'RO',
'SE01':'SE',
'SE02':'SE',
'SE03':'SE',
'SE04':'SE',
'SI00':'SI',
'SK00':'SK',
'UK00':'GB',
'UKNI':'GB'
}

## Load data

In [15]:
#load sheet with reference grid
df_in = pd.read_excel(fn_in, sheet_name="1. Elec Ref Grid")
df_in.columns = ["border", "ntc", "ntc2"]
df_in.head()

,border,ntc,ntc2
0,AL00-GR00,600,600
1,AL00-ME00,400,400
2,AL00-MK00,500,500
3,AL00-RS00,250,250
4,AT00-CH00,1200,1200


In [16]:
# new data frame with split value columns
new = df_in["border"].str.split("-", n=1, expand=True)
# making separate first name column from new data frame
df_in["from"] = new[0]
# making separate last name column from new data frame
df_in["to"] = new[1]
df_in.head()

,border,ntc,ntc2,from,to
0,AL00-GR00,600,600,AL00,GR00
1,AL00-ME00,400,400,AL00,ME00
2,AL00-MK00,500,500,AL00,MK00
3,AL00-RS00,250,250,AL00,RS00
4,AT00-CH00,1200,1200,AT00,CH00


In [ ]:
#we only keep the higher NTC value
df_in=df_in[['from','to','ntc']]

In [21]:
#rename and aggregate
df_in['from'] = df_in['from'].map(dict_agg_country)
df_in['to'] = df_in['to'].map(dict_agg_country)
df_in = df_in.dropna()
df_in_agg = df_in.groupby(['from','to']).sum().reset_index()
df_in_agg = df_in_agg[df_in_agg['from'] != df_in_agg['to']]
df_in_agg.head()

,from,to,ntc
0,AT,CH,1200
1,AT,CZ,900
2,AT,DE,7500
3,AT,HU,800
4,AT,IT,875


## Upsample data

Given values are static. So we assign a random date and upsample them to hourly frequency

In [8]:
start = pd.to_datetime("2024/01/01 00:00")
end = pd.to_datetime("2025/01/01 00:00")
df_start = df_in_agg.copy()
df_end = df_in_agg.copy()
df_start["date"] = start
df_end["date"] = end
df_ntc_in = pd.concat([df_start, df_end])
df_ntc_in = df_ntc_in.set_index(["date", "from", "to"])

In [9]:
df_ntc_in.head()

ntc
date       from to      
2024-01-01 AT   CH  1200
                CZ   900
                DE  7500
                HU   800
                IT   875

In [10]:
df_ntc = df_ntc_in.unstack().unstack().resample("h").ffill()
#df_ntc.index = df_ntc.index.tz_localize("utc")
df_ntc = df_ntc.stack().stack().reset_index()
df_ntc.head()

C:\Users\jonas\AppData\Local\Temp\ipykernel_23756\772011672.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_ntc = df_ntc.stack().stack().reset_index()
C:\Users\jonas\AppData\Local\Temp\ipykernel_23756\772011672.py:3: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_ntc = df_ntc.stack().stack().reset_index()


,date,from,to,ntc
0,2024-01-01,AT,CH,1200.0
1,2024-01-01,AT,CZ,900.0
2,2024-01-01,AT,DE,7500.0
3,2024-01-01,AT,HU,800.0
4,2024-01-01,AT,IT,875.0


## Export

In [11]:
df_ntc.to_csv(fn_out, encoding="utf-8", index=False)